# Tutorial 4: Density estimation using datasets with more than 600.000 cells

## Introduction

This tutorial adapts Tutorial 1 to compute the log-probability density function of single-cell flow cytometry datasets with more than 600,000 cells. No such file was used in our paper, so we assume access to a CSV file that pre-processed with logicle transformation and z-scored normalized.

In [9]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
from sklearn.model_selection import ParameterGrid
import os
import sys

sys.path.append(os.path.abspath("../src/TimeFlow/"))

import dataset
from dataset import CytometryData
import density
from density import DensityEstimation
import train
from train import Training

In [10]:
np.random.seed(1)
torch.manual_seed(1)

In [12]:
# Load the CSV file with pre-processed and scaled data
data = pd.read_csv("../Pre-processed-datasets/large_file.csv")
# We exclude the cell labels, which are not used during density estimation and pseudotime computation
# assuming 20 CD markers 
data = data.iloc[:,:20]

In [14]:
# Create dataset instances
# Training set
dataset_train = CytometryData(data, mode='train')
# Validation set
dataset_val = CytometryData(data, mode='val')     
# Test set
# dataset_test = CytometryData(data, mode='test')  

In [15]:
# Define the hyper-parameters 
hyper_parameters = {
    'coupling_layers': [10],
    'learning_rate': [0.001],
    "hidden_units": [256],
    "batch_size": [512]
}
grid = list(ParameterGrid(hyper_parameters))

Train the Real NVP model.

In [20]:
# Store hyper-parameters and validation loss
results_list = []

# Input data dimension
D = 20   

# 
for idx, hyper_parameters in enumerate(grid):
    D = dataset_train.data.shape[1]
        # Splits input dimensions for data partitions  
    D_half = D // 2
    if D % 2 == 0:
        input_dims = output_dims = D_half
    else:
        input_dims = D_half + 1
        output_dims = D_half

for idx, params in enumerate(grid):

    training_loader = DataLoader(dataset_train, batch_size=params['batch_size'], shuffle=False)
    val_loader = DataLoader(dataset_val, batch_size=params['batch_size'], shuffle=False)
    #test_loader = DataLoader(dataset_test, batch_size=params['batch_size'], shuffle=False)

    # Scaling (s) neural networks
    scaling_nn = lambda: nn.Sequential(
        nn.Linear(input_dims, params['hidden_units']),
        nn.LeakyReLU(),
        nn.Linear(params['hidden_units'], params['hidden_units']),
        nn.LeakyReLU(),
        nn.Linear(params['hidden_units'], output_dims),
        nn.Tanh()
    )

    # Shifting (t) neural networks
    shifting_nn = lambda: nn.Sequential(
        nn.Linear(input_dims, params['hidden_units']),
        nn.LeakyReLU(),
        nn.Linear(params['hidden_units'], params['hidden_units']),
        nn.LeakyReLU(),
        nn.Linear(params['hidden_units'], params['hidden_units']), nn.LeakyReLU(),
        nn.Linear(params['hidden_units'], output_dims)
    )
    
    
    # Base distribution set to multivariate standard Normal 
    z_dist = torch.distributions.MultivariateNormal(torch.zeros(D), torch.eye(D))
    
    # Real NVP model 
    model = density.DensityEstimation(D, z_dist, hyper_parameters['coupling_layers'], shifting_nn, scaling_nn)
    optimizer = torch.optim.Adamax([p for p in model.parameters() if p.requires_grad == True], lr=hyper_parameters['learning_rate'])

    # Directory to store the model weights and the density values
    save_dir = "./Results-large_file/"
    
    os.makedirs(save_dir, exist_ok=True)
    
    # save_path = os.path.join(save_dir, model_name)
    # torch.save(model, save_path)
    
    name = os.path.join(save_dir, "Large_file_Real_NVP_runtime")

    # Number of epochs for training
    num_epochs = 500

    # Maximum patience for early stopping
    max_patience = 15 

    # Initializes the Training class
    training_instance = Training(name, max_patience, num_epochs, model, optimizer, training_loader, val_loader, D)

    # Trains the model
    nll_val, epochs_trained = training_instance.train()
    
    # Number of epochs need to complete training
    print(f'Training completed after {epochs_trained} epochs.')

    results_list.append({
        'params': hyper_parameters,
        #'test_loss': test_loss,
        "val_loss": nll_val,
        #"epochs_needed": epochs_needed
    })


Loss improved, new model is saved.
Loss improved, new model is saved.
Loss improved, new model is saved.
Loss improved, new model is saved.
Loss improved, new model is saved.
Loss improved, new model is saved.
Loss improved, new model is saved.
Loss improved, new model is saved.
Loss improved, new model is saved.
Loss improved, new model is saved.
Loss improved, new model is saved.
Loss improved, new model is saved.
Loss improved, new model is saved.
Loss improved, new model is saved.
Loss improved, new model is saved.
Loss improved, new model is saved.
Loss improved, new model is saved.
Loss improved, new model is saved.
Loss improved, new model is saved.
Loss improved, new model is saved.
Loss improved, new model is saved.
Loss improved, new model is saved.
Training completed after 44 epochs.


Evaluate the probability density function at every data point (cell) of the large dataset.

In [ ]:
import os
import pandas as pd
import torch
from torch.utils.data import DataLoader
import numpy as np

# Directories for models and results
models_directory = "C://Users/Margarita/GitHub-TimeFlow/Results-large_file/"
results_directory = "C://Users/Margarita/GitHub-TimeFlow/Results-large_file/"

if not os.path.exists(results_directory):
    os.makedirs(results_directory)

# Define chunk size for reading CSV, can be adjusted
chunksize = 300000  

for model_file in os.listdir(models_directory):
    
    if model_file.endswith(".model"):
        model_path = os.path.join(models_directory, model_file)
        model = torch.load(model_path)
        model.eval()

        all_nll_results = []

        # Process each chunk
        for chunk in pd.read_csv("./Pre-processed-datasets/large_file.csv", chunksize=chunksize):

            # Chunk to tensor
            data_for_pdf = torch.tensor(chunk.iloc[:, :20].to_numpy().astype(np.float32))

            # Chunk dataloader
            evalLoader = DataLoader(data_for_pdf, batch_size=300000, shuffle=False)

            # Chunk processing
            with torch.no_grad():
                for batch in evalLoader:
                    output_pdf = model.log_probability_outputs(batch)

            output_pdf_arr = output_pdf.detach().numpy()

            all_nll_results.append(output_pdf_arr)

        # Chunk concatenation
        all_nll_results = np.concatenate(all_nll_results, axis=0)

        nll_results = pd.DataFrame(all_nll_results, columns=["pdf"])

        save_path = os.path.join(results_directory, f"density_large_file_{model_file}.csv")
        nll_results.to_csv(save_path, index=False)

        print(f"Results saved for model: {model_file}")


In [24]:
nll_results

,pdf
0,-19.902372
1,-19.294424
2,-17.888763
3,-26.869480
4,-24.836872
...,...
39251,-19.336288
39252,-23.642403
39253,-25.377218
39254,-26.672804


In [22]:
results_list

[{'params': {'batch_size': 512,
   'coupling_layers': 10,
   'hidden_units': 256,
   'learning_rate': 0.001},
  'val_loss': array([-18.42873789, -20.53990795, -21.35018664, -21.95961977,
         -22.42128629, -22.77094327, -23.07466864, -23.31068644,
         -23.42667563, -23.53709735, -23.58274717, -23.63933787,
         -23.65756715, -23.59989797, -23.66579537, -23.84350221,
         -24.01235373, -24.11671415, -24.12388113, -24.16444314,
         -24.18479281, -24.09253261, -23.92645803, -23.88138074,
         -23.84812994, -23.96849881, -24.15503353, -24.27462453,
         -24.33047452, -24.28720035, -24.26129507, -23.9720114 ,
         -24.13711108, -24.19782041, -24.24002987, -24.1581668 ,
         -24.09126423, -23.75038844, -23.94719572, -24.05157753,
         -24.14152668, -24.25532117, -24.28966812, -24.24060996,
         -24.22942244])}]